In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv
import os
load_dotenv(override = True)
print("HTTPS_PROXY:",os.environ.get("HTTPS_PROXY"))
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash-0731",
)

HTTPS_PROXY: http://127.0.0.1:7897


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware,ToolRetryMiddleware,ModelRetryMiddleware,SummarizationMiddleware
from langchain.tools.tool_node import ToolCallRequest

# 工具报错中间件
def on_error(exception:Exception,request:ToolCallRequest) -> str | None:
    if isinstance(exception,ValueError):
        return f"`{request.tool_call['name']}` failed with {type(exception).__name__}"

@tool
def get_weather(city:str) -> str:
    """获取指定城市的天气信息"""
    try:
        return f"明天{city}的天气是晴天"
    except Exception as e:
        raise ValueError(f"获取天气信息失败: {e}")

agent = create_agent(
    model = model,
    tools = [get_weather],
    middleware = [
        ToolErrorMiddleware(on_error=on_error),
        ToolRetryMiddleware(max_retries=3,backoff_factor=2,initial_delay=1),    # 最大重试次数，避退银子，初始等待
        ModelRetryMiddleware(max_retries=3,backoff_factor=2,initial_delay=1),
        SummarizationMiddleware(model="openrouter:deepseek/deepseek-v4-flash-0731",trigger=("token",4000),keep=("messages",10)), # 4000token时触发 保留10轮会话
    ],
)

result = agent.invoke({
    "messages":[{"role":"user","content":"What is weather in Beijing?"}]
})
print(result)

{'messages': [HumanMessage(content='What is weather in Beijing?', additional_kwargs={}, response_metadata={}, id='8f65d566-8b18-4bcc-a09d-221e88841500'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks about weather in Beijing. I should call the get_weather tool with city "Beijing".', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user asks about weather in Beijing. I should call the get_weather tool with city "Beijing".'}]}, response_metadata={'model_name': 'deepseek/deepseek-v4-flash-0731', 'id': 'gen-1789723470-Zzzw4eDCSTS7sryaeX8c', 'created': 1789723470, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 3.3481712e-05, 'cost_details': {'upstream_inference_completions_cost': 1.1974776e-05, 'upstream_inference_prompt_cost': 2.1506936e-05, 'upstream_inference_cost': 3.3481712e-05}}, id='lc_run--01a0b3d4-e874-78f3-a17b-84af8afd340b-0', tool_ca

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInputMiddleware
from langgraph.checkpoint.memory import MemorySaver

def read_email_tool(email_id:str) ->str:
    """读取指定邮箱的邮件内容"""
    return f"Email content for ID: {email_id}"
